In [1]:
# how to uncover truths that don't matter - second section

In [2]:
# loading libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
import time

In [4]:
# import functions
import coriolis_functions 
# import selfbuild module
import coriolis_module
# check out if import worked as expected
print(dir(coriolis_module))
print(dir(coriolis_functions))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_drift_distances', 'calculate_velocities', 'coriolis_acc', 'direction_vector', 'haversine', 'radius_at_latitude', 'rotation_matrix']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calcpp_total_drift', 'calculate_total_drift', 'coriolis_acc', 'coriolis_integral', 'coriolis_module', 'direction_vector', 'haversine', 'main', 'np', 'radius_at_latitude', 'rotation_matrix']


In [5]:
org_fdf = pd.read_csv("data/flights_sample_3m.csv").astype(
    {
    "FL_DATE": "datetime64[ns]", 
    "AIRLINE": "category", 
    "AIRLINE_DOT": "category", 
    "AIRLINE_CODE": "category", 
    "ORIGIN": "category", 
    "ORIGIN_CITY": "category", 
    "DEST": "category",
    "DEST_CITY": "category", 
    "CANCELLED": "bool",
    "CANCELLATION_CODE": "category", 
    "DIVERTED": "bool",
    }
)
ldf = pd.read_csv("data/airports.csv")

In [6]:
print(f"{len(org_fdf)} : rows in dataset")
print(org_fdf.dtypes)

3000000 : rows in dataset
FL_DATE                    datetime64[ns]
AIRLINE                          category
AIRLINE_DOT                      category
AIRLINE_CODE                     category
DOT_CODE                            int64
FL_NUMBER                           int64
ORIGIN                           category
ORIGIN_CITY                      category
DEST                             category
DEST_CITY                        category
CRS_DEP_TIME                        int64
DEP_TIME                          float64
DEP_DELAY                         float64
TAXI_OUT                          float64
WHEELS_OFF                        float64
WHEELS_ON                         float64
TAXI_IN                           float64
CRS_ARR_TIME                        int64
ARR_TIME                          float64
ARR_DELAY                         float64
CANCELLED                            bool
CANCELLATION_CODE                category
DIVERTED                             bool
CRS_ELAP

In [7]:
# setting up primary keys for later merge
org_fdf["primindex"] = org_fdf.index

In [8]:
# filtering dataframe

In [9]:
# dropping cancelled flights
fdf = org_fdf[~org_fdf["CANCELLED"]]

In [10]:
# droppting for calculations irrelevant columns
fdf = fdf.drop( ['AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN_CITY', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT',        
       'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'] ,axis=1)

In [11]:
# calculation of airtime based on time between wheels off and wheels on timestamps

# defining a helper function
def convert_time(df, time_columns):
    """ Convert integer time columns to HH:MM format """
    for col in time_columns:
        # Handle missing values and convert times
        df[col] = df[col].fillna(0).astype(int).apply(lambda x: f"{x//100:02d}:{x%100:02d}")
        # Adjust for hours == 24
        df[col] = df[col].replace("24:00", "00:00")
    return df

# applying the time conversion
fdf = convert_time(fdf, ["WHEELS_OFF", "WHEELS_ON"])

# setting up datetime columns for wheels off and wheels on
fdf["woff_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_OFF"])
fdf["won_datetime"] = pd.to_datetime(fdf["FL_DATE"].astype(str) + " " + fdf["WHEELS_ON"])

# calculating airtime based on time between wheels off and wheels on timestamps
fdf["airtime"] = fdf["won_datetime"] - fdf["woff_datetime"]

# dropping flights where airtime makes no sense
fdf = fdf[(fdf["airtime"].dt.total_seconds() > 0)]

In [12]:
# merging in geographical locations

In [13]:
# checking unique values in both datasets
fdf_airports = set(fdf["ORIGIN"].unique()).union(set(fdf["DEST"].unique()))
ldf_airports = set(ldf["IATA"].unique())  

# finding missing airport codes in ldf
missing_airports = fdf_airports.difference(ldf_airports)

# dropping rows where "ORIGIN" or "DEST" are in missing_airports
fdf = fdf[~fdf["ORIGIN"].isin(missing_airports) & ~fdf["DEST"].isin(missing_airports)]

# merging fdf with ldf to add geographical data for ORIGIN and DEST
fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")

fdf = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# dropping original origin and destination columns
fdf = fdf.drop(["ORIGIN", "DEST"], axis=1)

# enforcing categories on newly generated columns
fdf[["IATA_ORIGIN", "IATA_DEST"]] = fdf[["IATA_ORIGIN", "IATA_DEST"]].astype("category")

In [14]:
# calculations of drift distances

In [15]:
# calculating haversinedistance
fdf["haversine_distance"] = coriolis_functions.haversine(
    fdf["LATITUDE_ORIGIN"],
    fdf["LONGITUDE_ORIGIN"],
    fdf["LATITUDE_DEST"], 
    fdf["LONGITUDE_DEST"]
)

In [16]:
start_time = time.time()

In [17]:
fdf["average_velocity"] = fdf["haversine_distance"] / fdf["airtime"].dt.total_seconds()

In [18]:
# applying direction vector function row-wise
directions = fdf.apply(
    lambda row: coriolis_module.direction_vector(
        row["LATITUDE_ORIGIN"], 
        row["LONGITUDE_ORIGIN"], 
        row["LATITUDE_DEST"], 
        row["LONGITUDE_DEST"]
    ), axis=1
)

# Split the resulting direction vectors into separate columns
fdf["x_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[0])
fdf["y_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[1])
fdf["z_direction"] = fdf["average_velocity"] * directions.apply(lambda x: x[2])

In [19]:
# applying the calculate_total_drift function row by row
fdf["total_drift_distance"] = fdf.apply(
    lambda row: coriolis_functions.calcpp_total_drift(
        row, row["airtime"].total_seconds(), num_steps=100  # adjust num_steps for precision
    ), axis=1
)

In [20]:
end_time = time.time()
print(f"{end_time - start_time:,.2f} : calculation time in seconds")

1,662.29 : calculation time in seconds


In [21]:
print(f"{len(fdf):,} : Rows in dataset at time of evaluation.")

2,730,145 : Rows in dataset at time of evaluation.


In [22]:
# merging the cleaned df with the original df using primindex
final_fdf = org_fdf.merge(fdf[["primindex", "total_drift_distance"]], on="primindex", how="left")

In [23]:
print(f"{len(final_fdf)} : Rows in dataset after merging with the original.")

3000000 : Rows in dataset after merging with the original.


In [24]:
final_fdf = final_fdf.drop(["primindex"], axis=1)

In [25]:
print(f"Names of columns of the dataframe: \n", final_fdf.columns)

Names of columns of the dataframe: 
 Index(['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT', 'total_drift_distance'],
      dtype='object')


In [26]:
final_fdf.to_csv("data/coriolis_flights_3m.csv", index=False, sep=";", quoting=1)